# 基盤パッケージの更新

OS提供の基盤パッケージをホストごとに更新する。対象ごとに作業コピーを作る。

## *Operation Note*

*This is a cell for your own recording. ここに経緯を記述*

# 操作対象クラスタの設定

D10で作成したinventoryとgroup_varsを使用する。

In [ ]:
target_group = 'hadoop_all_testcluster'
target_group

In [ ]:
!ansible {target_group} -m ping

In [ ]:
import yaml
with open('group_vars/' + target_group) as configuration:
    kerberos_enabled = 'kerberos_realm' in yaml.safe_load(configuration)
kerberos_enabled

# 管理操作の認証

Kerberos構成の場合、HDFSの管理操作を行うホストでhdfsユーザーのkeytabを使って認証する。

In [ ]:
if kerberos_enabled:
    !ansible 'hadoop_namenode:hadoop_client' -l {target_group} --become --become-user hdfs -m shell -a 'kinit -kt /etc/hadoop/conf/hdfs.keytab hdfs/$(hostname -f)'

# Playbookの準備

In [ ]:
import os
import tempfile
work_dir = tempfile.mkdtemp()
work_dir

使用するNotebookと同じ版のPlaybookを取得する。

In [ ]:
!git clone https://github.com/NII-cloud-operation/Literate-computing-Hadoop.git {work_dir}/hadoop
!git -C {work_dir}/hadoop log -1

In [ ]:
playbook_dir = os.path.join(work_dir, 'hadoop/playbooks')
playbook_dir

In [ ]:
upgrade_host = 'XXX.XXX.XXX.114'
upgrade_host

In [ ]:
!ansible {upgrade_host} -m debug -a var=group_names -l {target_group}

inventoryの役割に従って、KDCのバックアップとZooKeeperの起動確認を行う。

# 更新内容の確認

In [ ]:
!ansible {upgrade_host} --become -m apt -a "update_cache=yes" -l {target_group}

In [ ]:
!ansible {upgrade_host} --become -CDv -m apt -a "name=openjdk-11-jdk,cgroup-tools,cgroup-lite state=latest only_upgrade=yes" -l {target_group}

In [ ]:
if kerberos_enabled:
    !ansible {upgrade_host} --become -CDv -m apt -a "name=krb5-kdc,krb5-admin-server,krb5-kpropd,krb5-user,krb5-config state=latest only_upgrade=yes" -l {target_group}

更新対象と変更内容を確認する。

KDCのバックアップはrootのみが読める保管先に作成する。

In [ ]:
!ansible 'kerberos_server:&{upgrade_host}:&{target_group}' --become -m shell -a "umask 077; kdb5_util dump /var/lib/krb5kdc/pre-upgrade.dump"

# 利用停止

利用者のジョブ投入とHBaseクライアントを停止し、実行中ジョブの終了を待つ。

In [ ]:
!ansible hadoop_client -a "yarn application -list -appStates SUBMITTED,ACCEPTED,RUNNING" -l {target_group}

実行中アプリケーションが0件であることを確認する。

In [ ]:
!ansible-playbook {playbook_dir}/stop_hbase_regionserver.yml -l {target_group}

In [ ]:
!ansible-playbook {playbook_dir}/stop_hbase_master.yml -l {target_group}

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs haadmin -getAllServiceState' -l {target_group}

対象にActive NameNodeがある場合、HAの現在の出力から切替先を指定してフェイルオーバーする。

In [ ]:
platform_active_id = 'nn1'
platform_standby_id = 'nn2'
platform_failover = False
(platform_active_id, platform_standby_id, platform_failover)

In [ ]:
if platform_failover:
    !ansible hadoop_client --become --become-user hdfs -a "hdfs haadmin -failover {platform_active_id} {platform_standby_id}" -l {target_group}

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs dfsadmin -report' -l {target_group}

Live datanodesが対象の全台であり、Missing blocks・Blocks with corrupt replicas・Under replicated blocksがすべて0であることを確認する。回復中なら時間をおいて再確認する。

# 対象ホストのサービス停止

Active NameNodeが他ホストへ切り替わり、ZooKeeper・JournalNodeの過半数を維持できることを確認する。O12bの停止順で、この1台だけを停止する。

In [ ]:
!ansible-playbook {playbook_dir}/stop_spark_historyserver.yml -l {upgrade_host}

In [ ]:
!ansible-playbook {playbook_dir}/stop_mapreduce_historyserver.yml -l {upgrade_host}

In [ ]:
!ansible-playbook {playbook_dir}/stop_timelineservice.yml -l {upgrade_host}

In [ ]:
!ansible-playbook {playbook_dir}/stop_nodemanager.yml -l {upgrade_host}

In [ ]:
!ansible-playbook {playbook_dir}/stop_resourcemanager.yml -l {upgrade_host}

In [ ]:
!ansible-playbook {playbook_dir}/stop_namenode.yml -l {upgrade_host}

In [ ]:
!ansible-playbook {playbook_dir}/stop_datanode.yml -l {upgrade_host}

In [ ]:
!ansible-playbook {playbook_dir}/stop_journalnode.yml -l {upgrade_host}

In [ ]:
!ansible-playbook {playbook_dir}/stop_zookeeper-server.yml -l {upgrade_host}

# パッケージの更新

In [ ]:
!ansible {upgrade_host} --become -m apt -a "name=openjdk-11-jdk,cgroup-tools,cgroup-lite state=latest only_upgrade=yes" -l {target_group}

In [ ]:
if kerberos_enabled:
    !ansible {upgrade_host} --become -m apt -a "name=krb5-kdc,krb5-admin-server,krb5-kpropd,krb5-user,krb5-config state=latest only_upgrade=yes" -l {target_group}

# 対象ホストのサービス起動

更新されたJava等を使って稼働するよう、O12aの起動順でサービスを戻す。

In [ ]:
!ansible-playbook {playbook_dir}/start_zookeeper-server.yml -l {upgrade_host}

In [ ]:
!ansible 'hadoop_zookeeperserver:&{upgrade_host}:&{target_group}' -m wait_for -a "host={upgrade_host} port=2181 timeout=120"

In [ ]:
!ansible-playbook {playbook_dir}/start_journalnode.yml -l {upgrade_host}

In [ ]:
!ansible-playbook {playbook_dir}/start_namenode.yml -l {upgrade_host}

In [ ]:
!ansible-playbook {playbook_dir}/start_datanode.yml -l {upgrade_host}

In [ ]:
!ansible-playbook {playbook_dir}/start_resourcemanager.yml -l {upgrade_host}

In [ ]:
!ansible-playbook {playbook_dir}/start_nodemanager.yml -l {upgrade_host}

In [ ]:
!ansible-playbook {playbook_dir}/start_timelineservice.yml -l {upgrade_host}

In [ ]:
!ansible-playbook {playbook_dir}/start_mapreduce_historyserver.yml -l {upgrade_host}

In [ ]:
!ansible-playbook {playbook_dir}/start_spark_historyserver.yml -l {upgrade_host}

In [ ]:
!ansible-playbook {playbook_dir}/start_hbase_master.yml -l {target_group}

In [ ]:
!ansible-playbook {playbook_dir}/start_hbase_regionserver.yml -l {target_group}

In [ ]:
if kerberos_enabled:
    !ansible 'hadoop_namenode:hadoop_client' --become --become-user hdfs -m shell -a 'kinit -kt /etc/hadoop/conf/hdfs.keytab hdfs/$(hostname -f)' -l {target_group}

In [ ]:
!ansible {upgrade_host} -a "dpkg-query -W openjdk-11-jdk cgroup-tools cgroup-lite" -l {target_group}

In [ ]:
!ansible {upgrade_host} -a "java -version" -l {target_group}

In [ ]:
if kerberos_enabled:
    !ansible hadoop_client -m shell -a 'kinit -kt "$HOME/.hadoop.keytab" "$(id -un)"; klist' -l {target_group}

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs haadmin -getAllServiceState' -l {target_group}

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs dfsadmin -report' -l {target_group}

Live datanodesが対象の全台であり、Missing blocks・Blocks with corrupt replicas・Under replicated blocksがすべて0であることを確認する。回復中なら時間をおいて再確認する。

# サービス再開後のHBaseとYARNの利用確認

更新前データの保持と新しい書き込みを確認する。

In [ ]:
hbase_check_table = 'upgrade_check'
hbase_check_table

HBaseの管理操作はhbaseユーザーで行う。Kerberos構成では、サーバーの認証設定を使ってZooKeeperにも接続する。

In [ ]:
if kerberos_enabled:
    !ansible hadoop_hbase_master --become --become-user hbase -m shell -a 'kinit -kt /etc/hbase/conf/hbase.keytab hbase/$(hostname -f)' -l {target_group}

In [ ]:
hbase_query = f"""status 'detailed'
get '{hbase_check_table}', 'before'
put '{hbase_check_table}', 'after', 'data:value', 'writable'
get '{hbase_check_table}', 'after'
"""
with open(os.path.join(work_dir, "hbase-after.rb"), "w") as query_file:
    query_file.write(hbase_query)
print(hbase_query)

In [ ]:
!ansible hadoop_hbase_master --become -m copy -a "src={work_dir}/hbase-after.rb dest=/tmp/hbase-upgrade-after.rb owner=hbase group=hbase mode=0644" -l {target_group}

In [ ]:
if kerberos_enabled:
    !ansible hadoop_hbase_master --become --become-user hbase -a "hbase --auth-as-server shell -n /tmp/hbase-upgrade-after.rb" -l {target_group}
else:
    !ansible hadoop_hbase_master --become --become-user hbase -a "hbase shell -n /tmp/hbase-upgrade-after.rb" -l {target_group}

In [ ]:
!ansible hadoop_client -m shell -a "yarn jar /opt/hadoop/current/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar pi 2 1000" -l {target_group}

サービスと認証が復帰したことを確認して次のホストへ進む。全台が完了したら、[T12a](T12a_Hadoop%20-%20Confirm%20the%20services%20are%20alive%20-%20ZK,HDFS,YARN,HBase,Spark.ipynb)・[T12d](T12d_Hadoop%20-%20Simple%20Spark%20script%20for%20Test.ipynb)で確認して利用を再開する。